In [1]:
import os

print(os.getcwd())


c:\Users\germa\AppData\Local\Programs\Microsoft VS Code


In [ ]:
import pandas as pd

# Loading 
df = pd.read_csv("NASDAQ_PITON.csv")

# Convert Date column to datetime and sort in ascending order
df["Date"] = pd.to_datetime(df["Date"], format="%m/%d/%Y")
df = df.sort_values("Date").reset_index(drop=True)

# Initial investments
initial_qqq = 10000
initial_qqq3 = 10000

# Calculate cumulative growth factors
df["qqq_growth"] = (1 + df["qqq"]).cumprod()
df["qqq3_growth"] = (1 + df["qqq3"]).cumprod()

# Portfolio value over time
df["qqq_value"] = initial_qqq * df["qqq_growth"]
df["qqq3_value"] = initial_qqq3 * df["qqq3_growth"]
df["portfolio_value"] = df["qqq_value"] + df["qqq3_value"]

# Save results to CSV
df.to_csv("portfolio_values.csv", index=False)

# Final balance
final_balance = df["portfolio_value"].iloc[-1]
print(f"Final balance as of {df['Date'].iloc[-1].date()}: ${final_balance:,.2f}")
print("Detailed results saved to portfolio_values.csv")


Final balance as of NaT: $nan
Detailed results saved to portfolio_values.csv


In [ ]:
import pandas as pd

INPUT_CSV = "NASDAQ_PITON.csv"                    
OUTPUT_CSV = "portfolio_values_rebalanced.csv"  


df = pd.read_csv(INPUT_CSV)

df["Date"] = pd.to_datetime(df["Date"], format="%m/%d/%Y", errors="coerce")

df["qqq"] = pd.to_numeric(df["qqq"], errors="coerce")
df["qqq3"] = pd.to_numeric(df["qqq3"], errors="coerce")

bad_mask = df["Date"].isna() | df["qqq"].isna() | df["qqq3"].isna()
if bad_mask.any():
    bad = df.loc[bad_mask, ["Date", "qqq", "qqq3"]]
    print(f"Warning: dropping {len(bad)} rows with invalid/missing Date or returns. Example rows:")
    print(bad.head(5).to_string(index=False))
df = df.loc[~bad_mask].copy()


df = df.sort_values("Date").reset_index(drop=True)


initial_qqq = 7000.0
initial_qqq3 = 3000.0

qqq_value = float(initial_qqq)
qqq3_value = float(initial_qqq3)

records = []
rebalances = []

for row in df.itertuples(index=False):
    date = row.Date            
    qqq_ret = row.qqq          
    qqq3_ret = row.qqq3


    qqq_value *= (1.0 + qqq_ret)
    qqq3_value *= (1.0 + qqq3_ret)

    portfolio_value = qqq_value + qqq3_value
    qqq_ratio = qqq_value / portfolio_value if portfolio_value != 0 else 0.0

    rebalanced = False
    rebalance_reason = ""
    pre_qqq = qqq_value
    pre_qqq3 = qqq3_value

    if qqq_ratio <= 0.65 or qqq_ratio >= 0.75:
        qqq_value = portfolio_value * 0.70
        qqq3_value = portfolio_value * 0.30
        rebalanced = True
        rebalance_reason = "below_65" if qqq_ratio <= 0.6 else "above_75"
        rebalances.append({
            "Date": date.date(),
            "pre_qqq": pre_qqq,
            "pre_qqq3": pre_qqq3,
            "post_qqq": qqq_value,
            "post_qqq3": qqq3_value,
            "portfolio_value": portfolio_value,
            "reason": rebalance_reason
        })
        qqq_ratio = qqq_value / portfolio_value if portfolio_value != 0 else 0.0

    records.append({
        "Date": date.date(),
        "qqq_ret": qqq_ret,
        "qqq3_ret": qqq3_ret,
        "qqq_value": qqq_value,
        "qqq3_value": qqq3_value,
        "portfolio_value": qqq_value + qqq3_value,
        "qqq_ratio": qqq_ratio,
        "rebalanced": rebalanced,
        "rebalance_reason": rebalance_reason
    })


result_df = pd.DataFrame(records)
result_df.to_csv(OUTPUT_CSV, index=False)
final_date = result_df["Date"].iloc[-1]
final_balance = result_df["portfolio_value"].iloc[-1]

print(f"\nSimulation complete. Final balance as of {final_date}: ${final_balance:,.2f}")
print(f"Daily results written to: {OUTPUT_CSV}")
print(f"Total rebalances performed: {len(rebalances)}")
if rebalances:
    reb_df = pd.DataFrame(rebalances)
    print("\nFirst up to 5 rebalances:")
    print(reb_df.head(5).to_string(index=False))
    print("\nLast up to 5 rebalances:")
    print(reb_df.tail(5).to_string(index=False))


Date  qqq     qqq3
 NaT  NaN      NaN
 NaT  NaN      NaN
 NaT  NaN      NaN
 NaT  NaN 0.033413

Simulation complete. Final balance as of 2025-09-23: $12,574,454.60
Daily results written to: portfolio_values_rebalanced.csv
Total rebalances performed: 152

First up to 5 rebalances:
      Date      pre_qqq    pre_qqq3     post_qqq   post_qqq3  portfolio_value   reason
1985-11-27  7914.691072 4293.563202  8545.777992 3662.476282     12208.254274 above_75
1986-03-10  9670.652252 5237.038504 10435.383529 4472.307227     14907.690756 above_75
1986-04-25 11750.859298 6334.017259 12659.413590 5425.462967     18084.876557 above_75
1986-08-04 11232.254919 3705.319428 10456.302043 4481.272304     14937.574347 above_75
1987-01-15 12017.856128 6561.317217 13005.421342 5573.752004     18579.173346 above_75

Last up to 5 rebalances:
      Date      pre_qqq     pre_qqq3     post_qqq    post_qqq3  portfolio_value   reason
2024-08-05 5.915116e+06 1.899793e+06 5.470436e+06 2.344473e+06     7.814909e+06 ab

In [ ]:
import pandas as pd
import numpy as np
from itertools import product

INPUT_CSV = "NASDAQ_PITON.csv"
OUTPUT_CSV = "portfolio_values_best_rebalanced.csv"


df = pd.read_csv(INPUT_CSV)
df["Date"] = pd.to_datetime(df["Date"], format="%m/%d/%Y", errors="coerce")
df["qqq"] = pd.to_numeric(df["qqq"], errors="coerce")
df["qqq3"] = pd.to_numeric(df["qqq3"], errors="coerce")
df = df.dropna(subset=["Date", "qqq", "qqq3"]).sort_values("Date").reset_index(drop=True)


def run_backtest(lower_thr=0.5, upper_thr=0.7, 
                 target_qqq=0.7, initial_qqq=6000, initial_qqq3=4000):
    """
    Simulate portfolio growth with rebalancing when QQQ share leaves [lower_thr, upper_thr].
    Rebalance to target_qqq / (1 - target_qqq).
    """
    qqq_value = float(initial_qqq)
    qqq3_value = float(initial_qqq3)
    records = []

    for row in df.itertuples(index=False):
        date, qqq_ret, qqq3_ret = row.Date, row.qqq, row.qqq3

        qqq_value *= (1 + qqq_ret)
        qqq3_value *= (1 + qqq3_ret)

        portfolio_value = qqq_value + qqq3_value
        qqq_ratio = qqq_value / portfolio_value if portfolio_value != 0 else 0.0

        if qqq_ratio <= lower_thr or qqq_ratio >= upper_thr:
            qqq_value = portfolio_value * target_qqq
            qqq3_value = portfolio_value * (1 - target_qqq)
            qqq_ratio = qqq_value / portfolio_value

        records.append({
            "Date": date.date(),
            "qqq_value": qqq_value,
            "qqq3_value": qqq3_value,
            "portfolio_value": portfolio_value,
            "qqq_ratio": qqq_ratio
        })

    return pd.DataFrame(records)


best_value = -np.inf
best_params = None
best_df = None

target_qqq = 0.6

lower_range = np.arange(0.0, 0.5, 0.01)   # 0% → 60%
upper_range = np.arange(0.7, 1.0, 0.01)   # 60% → 100%


for lower, upper in product(lower_range, upper_range):
    if lower >= upper:
        continue 
    result = run_backtest(lower, upper, target_qqq=target_qqq)
    final_value = result["portfolio_value"].iloc[-1]

    if final_value > best_value:
        best_value = final_value
        best_params = (lower, upper)
        best_df = result


best_df.to_csv(OUTPUT_CSV, index=False)

print(f"\nBest thresholds found:")
print(f"  Lower: {best_params[0]*100:.0f}%   Upper: {best_params[1]*100:.0f}%")
print(f"Final portfolio value: ${best_value:,.2f}")
print(f"Results saved to: {OUTPUT_CSV}")



Best thresholds found:
  Lower: 2%   Upper: 96%
Final portfolio value: $355,201,260.22
Results saved to: portfolio_values_best_rebalanced.csv


In [ ]:
import pandas as pd
import numpy as np
from itertools import product

INPUT_CSV = "NASDAQ_PITON.csv"
OUTPUT_CSV = "portfolio_values_best_rebalanced.csv"


df = pd.read_csv(INPUT_CSV)
df["Date"] = pd.to_datetime(df["Date"], format="%m/%d/%Y", errors="coerce")
df["qqq"] = pd.to_numeric(df["qqq"], errors="coerce")
df["qqq3"] = pd.to_numeric(df["qqq3"], errors="coerce")
df = df.dropna(subset=["Date", "qqq", "qqq3"]).sort_values("Date").reset_index(drop=True)


def run_backtest(lower_thr=0.6, upper_thr=0.8, 
                 target_qqq=0.7, initial_qqq=7000, initial_qqq3=3000):
    """
    Simulate portfolio growth with rebalancing when QQQ share leaves [lower_thr, upper_thr].
    Records whether rebalance happened and the reason (low/high).
    """
    qqq_value = float(initial_qqq)
    qqq3_value = float(initial_qqq3)
    records = []

    for row in df.itertuples(index=False):
        date, qqq_ret, qqq3_ret = row.Date, row.qqq, row.qqq3

        qqq_value *= (1 + qqq_ret)
        qqq3_value *= (1 + qqq3_ret)

        portfolio_value = qqq_value + qqq3_value
        qqq_ratio = qqq_value / portfolio_value if portfolio_value != 0 else 0.0

        rebalanced = False
        rebalance_reason = ""

        if qqq_ratio <= lower_thr:
            qqq_value = portfolio_value * target_qqq
            qqq3_value = portfolio_value * (1 - target_qqq)
            qqq_ratio = qqq_value / portfolio_value
            rebalanced = True
            rebalance_reason = "low"
        elif qqq_ratio >= upper_thr:
            qqq_value = portfolio_value * target_qqq
            qqq3_value = portfolio_value * (1 - target_qqq)
            qqq_ratio = qqq_value / portfolio_value
            rebalanced = True
            rebalance_reason = "high"

        records.append({
            "Date": date.date(),
            "qqq_value": qqq_value,
            "qqq3_value": qqq3_value,
            "portfolio_value": portfolio_value,
            "qqq_ratio": qqq_ratio,
            "rebalanced": rebalanced,
            "rebalance_reason": rebalance_reason
        })

    return pd.DataFrame(records)


best_value = -np.inf
best_params = None
best_df = None


target_qqq = 0.6


lower_range = np.arange(0.0, 0.5, 0.01)  
upper_range = np.arange(0.7, 1.0, 0.01)   


for lower, upper in product(lower_range, upper_range):
    if lower >= upper:
        continue  
    result = run_backtest(lower, upper, target_qqq=target_qqq)
    final_value = result["portfolio_value"].iloc[-1]

    if final_value > best_value:
        best_value = final_value
        best_params = (lower, upper)
        best_df = result


best_df.to_csv(OUTPUT_CSV, index=False)

print(f"\nBest thresholds found:")
print(f"  Lower: {best_params[0]*100:.0f}%   Upper: {best_params[1]*100:.0f}%")
print(f"Final portfolio value: ${best_value:,.2f}")
print(f"Results saved to: {OUTPUT_CSV}")



Best thresholds found:
  Lower: 2%   Upper: 97%
Final portfolio value: $478,579,470.29
Results saved to: portfolio_values_best_rebalanced.csv


In [ ]:
import pandas as pd
import numpy as np
from itertools import product

INPUT_CSV = "NASDAQ_PITON.csv"
OUTPUT_CSV = "portfolio_values_all_combinations.csv"


df = pd.read_csv(INPUT_CSV)
df["Date"] = pd.to_datetime(df["Date"], format="%m/%d/%Y", errors="coerce")
df["qqq"] = pd.to_numeric(df["qqq"], errors="coerce")
df["qqq3"] = pd.to_numeric(df["qqq3"], errors="coerce")
df = df.dropna(subset=["Date", "qqq", "qqq3"]).sort_values("Date").reset_index(drop=True)

def run_backtest(lower_thr=0.6, upper_thr=0.8, 
                 target_qqq=0.7, initial_qqq=7000, initial_qqq3=3000):
    """
    Simulate portfolio growth with rebalancing when QQQ share leaves [lower_thr, upper_thr].
    Returns DataFrame with daily values and number of rebalances.
    """
    qqq_value = float(initial_qqq)
    qqq3_value = float(initial_qqq3)
    records = []
    rebalance_count = 0

    for row in df.itertuples(index=False):
        date, qqq_ret, qqq3_ret = row.Date, row.qqq, row.qqq3

        qqq_value *= (1 + qqq_ret)
        qqq3_value *= (1 + qqq3_ret)

        portfolio_value = qqq_value + qqq3_value
        qqq_ratio = qqq_value / portfolio_value if portfolio_value != 0 else 0.0

        rebalanced = False
        rebalance_reason = ""

        if qqq_ratio <= lower_thr:
            qqq_value = portfolio_value * target_qqq
            qqq3_value = portfolio_value * (1 - target_qqq)
            qqq_ratio = qqq_value / portfolio_value
            rebalanced = True
            rebalance_reason = "low"
        elif qqq_ratio >= upper_thr:
            qqq_value = portfolio_value * target_qqq
            qqq3_value = portfolio_value * (1 - target_qqq)
            qqq_ratio = qqq_value / portfolio_value
            rebalanced = True
            rebalance_reason = "high"

        if rebalanced:
            rebalance_count += 1

        records.append({
            "Date": date.date(),
            "qqq_value": qqq_value,
            "qqq3_value": qqq3_value,
            "portfolio_value": portfolio_value,
            "qqq_ratio": qqq_ratio,
            "rebalanced": rebalanced,
            "rebalance_reason": rebalance_reason
        })

    df_result = pd.DataFrame(records)
    return df_result, rebalance_count


target_qqq = 0.7 
lower_range = np.arange(0.0, 0.61, 0.01)   
upper_range = np.arange(0.6, 1.0, 0.01)

summary_records = []

for lower, upper in product(lower_range, upper_range):
    if lower >= upper:
        continue

    daily_df, rebalance_count = run_backtest(lower, upper, target_qqq=target_qqq)
    final_value = daily_df["portfolio_value"].iloc[-1]

    summary_records.append({
        "lower_thr": lower,
        "upper_thr": upper,
        "final_value": final_value,
        "num_rebalances": rebalance_count
    })


summary_df = pd.DataFrame(summary_records)
summary_df.to_csv(OUTPUT_CSV, index=False)

print(f"Results for all combinations saved to {OUTPUT_CSV}")


Results for all combinations saved to portfolio_values_all_combinations.csv
